In [ ]:
1+1

In [ ]:
import io
import numpy as np
from PIL import Image, ImageChops, ImageEnhance
import matplotlib.pyplot as plt

In [ ]:

def universal_ela_check(img_path: str, quality: int = 90, amplify: int = 20, threshold: float = 0.08):
    """
    Universal Error Level Analysis (ELA) for ALL image formats.
    """
    try:
        # 1. Open original
        original_img = Image.open(img_path)
        
        # 2. Handle transparency by flattening onto a white background
        if original_img.mode in ('RGBA', 'LA') or (original_img.mode == 'P' and 'transparency' in original_img.info):
            background = Image.new('RGB', original_img.size, (255, 255, 255))
            background.paste(original_img, mask=original_img.split()[-1])
            original = background
        else:
            original = original_img.convert("RGB")

        # 3. Save to memory as JPEG at fixed quality
        buffer = io.BytesIO()
        original.save(buffer, "JPEG", quality=quality)
        buffer.seek(0)
        recompressed = Image.open(buffer).convert("RGB")

        # 4. Calculate absolute difference
        diff = ImageChops.difference(original, recompressed)

        # 5. Amplify the difference
        enhancer = ImageEnhance.Brightness(diff)
        amplified = enhancer.enhance(amplify)

        # 6. Calculate score
        diff_array = np.array(amplified)
        score = float(np.mean(diff_array) / 255.0)

        return {
            "is_suspicious": score > threshold,
            "ela_score": round(score, 4),
            "threshold": threshold,
            "ela_image": amplified,       # Returned for visual testing
            "original_image": original    # Returned for visual testing
        }
    except Exception as e:
        return {"error": str(e)}


In [ ]:
import logging
from PIL import Image
from PIL.ExifTags import TAGS
import cv2
import numpy as np

# Configure logging to track forensic steps
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

class ImageForensicsAnalyzer:
    def __init__(self, software_signatures=None):
        self.signatures = software_signatures or [
            'adobe', 'photoshop', 'gimp', 'canva', 'firefly',
            'stability', 'midjourney', 'dall-e', 'stable diffusion'
        ]

    def extract_metadata_signatures(self, file_path: str) -> bool:
        """Inspects both file info blocks (PNG) and EXIF data for editing/AI signatures."""
        logger.info(f"Executing Metadata Analysis on: {file_path}")
        try:
            with Image.open(file_path) as img:
                # 1. Check img.info (crucial for PNG text chunks, software tags, embedded prompts)
                for key, value in img.info.items():
                    if any(sig in str(value).lower() for sig in self.signatures):
                        logger.warning(f"Metadata signature hit found: {key} -> {value}")
                        return True

                # 2. Check EXIF data (standard for JPEGs and traditional image formats)
                exif_data = img._getexif()
                if exif_data:
                    for tag_id, value in exif_data.items():
                        tag_name = TAGS.get(tag_id, tag_id)
                        if tag_name == 'Software' and any(
                            sig in str(value).lower() for sig in self.signatures
                        ):
                            logger.warning(f"EXIF tag signature hit found: Software -> {value}")
                            return True
                            
            logger.info("Metadata verification passed clean.")
            return False
            
        except Exception as e:
            logger.error(f"Could not parse metadata for {file_path}: {e}")
            return False

    def analyze_image_comprehensive(self, file_path: str) -> dict:
        """Runs a complete check combining metadata footprints and pixel anomalies."""
        report = {
            "file_path": file_path,
            "is_suspicious": False,
            "flags": []
        }

        # Step 1: Metadata Signature Check
        has_bad_metadata = self.extract_metadata_signatures(file_path)
        if has_bad_metadata:
            report["is_suspicious"] = True
            report["flags"].append("Editing/AI software signature detected in metadata.")

        # Step 2: Pixel Noise & Edge Variance Check (OpenCV fallback for stripped metadata)
        cv_img = cv2.imread(file_path)
        if cv_img is not None:
            gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
            laplacian = cv2.Laplacian(gray, cv2.CV_64F)
            
            # Local block variation check for pasted sections
            h, w = gray.shape
            block_size = 64
            block_variances = []
            
            for y in range(0, h - block_size, block_size):
                for x in range(0, w - block_size, block_size):
                    block = laplacian[y:y+block_size, x:x+block_size]
                    block_variances.append(np.var(block))
                    
            if block_variances:
                var_std = np.std(block_variances)
                if var_std > (np.mean(block_variances) * 1.5):
                    report["is_suspicious"] = True
                    report["flags"].append("Localized texture/noise inconsistency detected (possible paste/splice).")

        return report

# ==========================================
# 🧪 EXECUTION
# ==========================================
if __name__ == "__main__":
    analyzer = ImageForensicsAnalyzer()
    test_path = r"D:\Jamaludeen\Project-1-AI-Image-detection\images\manual_Edited_img.png"
    
    result = analyzer.analyze_comprehensive(test_path)
    print("\n--- Final Forensic Report ---")
    print(f"🚨 Is Suspicious: {result['is_suspicious']}")
    for flag in result['flags']:
        print(f"   - {flag}")

In [ ]:
import cv2
import numpy as np
from PIL import Image
from PIL.ExifTags import TAGS

def analyze_image_forensics(image_path):
    print(f"[*] Running forensic analysis on: {image_path}\n")
    report = {
        "is_suspicious": False,
        "flags": [],
        "software_detected": None,
        "details": {}
    }
    
    try:
        # 1. Load image using PIL for metadata and OpenCV for pixel analysis
        pil_img = Image.open(image_path)
        cv_img = cv2.imread(image_path)
        
        if cv_img is None:
            return {"error": "Could not read image file with OpenCV."}

        # ==========================================
        # Check 1: EXIF & Software Metadata Analysis
        # ==========================================
        exifdata = pil_img.getexif()
        software = None
        
        for tag_id in exifdata:
            tag = TAGS.get(tag_id, tag_id)
            if tag == 'Software':
                software = exifdata.get(tag_id)
                break
                
        if software:
            report["software_detected"] = software
            report["flags"].append(f"Editing software signature found in metadata: {software}")
            report["is_suspicious"] = True
        else:
            report["details"]["metadata"] = "No editing software signature found (metadata may have been stripped)."

        # ==========================================
        # Check 2: Noise & High-Frequency Discontinuity (Pasted Object Detection)
        # ==========================================
        # Convert to grayscale
        gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
        
        # Apply Laplacian operator to detect sharp edges and noise variance
        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        variance = np.var(laplacian)
        
        report["details"]["noise_variance"] = float(variance)
        
        # Localized variance check (sliding block analysis for pasted regions)
        h, w = gray.shape
        block_size = 64
        block_variances = []
        
        for y in range(0, h - block_size, block_size):
            for x in range(0, w - block_size, block_size):
                block = laplacian[y:y+block_size, x:x+block_size]
                block_variances.append(np.var(block))
                
        # If variance standard deviation across blocks is abnormally high, 
        # it indicates inconsistent textures/pasted elements.
        var_std = np.std(block_variances)
        report["details"]["texture_inconsistency_score"] = float(var_std)
        
        if var_std > (np.mean(block_variances) * 1.5):
            report["flags"].append("Localized texture/noise inconsistency detected (possible pasted/spliced region).")
            report["is_suspicious"] = True

        # ==========================================
        # Check 3: Format & Compression Sanity Check
        # ==========================================
        if pil_img.format == 'PNG':
            report["details"]["format_note"] = "Image is PNG (Lossless). Standard ELA is ineffective; relying on noise and metadata analysis."

    except Exception as e:
        return {"error": str(e)}

    return report

# ==========================================
# 🧪 EXECUTION
# ==========================================
if __name__ == "__main__":
    # Use a raw string (r"...") to avoid path escape warnings
    test_image_path = r"D:\Jamaludeen\Project-1-AI-Image-detection\images\manual_Edited_img.jpg"
    
    result = analyze_image_forensics(test_image_path)
    
    if "error" in result:
        print("❌ Error:", result["error"])
    else:
        print(f"🚨 Is Suspicious: {result['is_suspicious']}")
        if result['software_detected']:
            print(f"🛠️ Software Signature: {result['software_detected']}")
        print("🚩 Flags Raised:")
        for flag in result["flags"]:
            print(f"   - {flag}")
        print("\n📊 Technical Details:", result["details"])

In [ ]:
import cv2
import numpy as np

def detect_copy_paste_forgery(image_path):
    print(f"[*] Scanning for duplicated/pasted regions in: {image_path}")
    
    # Read the image in grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return "Error: Could not load image."

    # Initialize ORB detector (Prebuilt OpenCV feature extractor)
    orb = cv2.ORB_create(nfeatures=2000)
    keypoints, descriptors = orb.detectAndCompute(img, None)

    if descriptors is None or len(descriptors) < 2:
        return "Not enough features found in the image to analyze."

    # Use Brute-Force Matcher to find similar patches within the same image
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    matches = bf.knnMatch(descriptors, descriptors, k=2)

    # Apply Ratio Test to filter true matches from background noise
    good_matches = []
    for match_pair in matches:
        if len(match_pair) == 2:
            m, n = match_pair
            # If the distance ratio is tight, it means a pattern repeats
            if m.distance < 0.4 * n.distance:
                # Ensure it's not matching the exact same pixel coordinates (self-match)
                pt1 = keypoints[m.queryIdx].pt
                pt2 = keypoints[m.trainIdx].pt
                distance_between_points = np.sqrt((pt1[0] - pt2[0])**2 + (pt1[1] - pt2[1])**2)
                
                if distance_between_points > 20: # Must be a duplicated region elsewhere
                    good_matches.append(m)

    print(f"[*] Found {len(good_matches)} duplicated feature clusters.")
    
    if len(good_matches) > 15: # Arbitrary threshold for suspicious repetition
        print("🚨 SUSPICIOUS: High number of duplicated patterns detected (Possible copy-paste or clone tool usage).")
        return True
    else:
        print("✅ Clean: No significant duplicated patterns found.")
        return False

# Run the prebuilt package test
detect_copy_paste_forgery(r"D:\Jamaludeen\Project-1-AI-Image-detection\images\manual_Edited_img.jpg")

In [9]:
import logging
from PIL import Image
from PIL.ExifTags import TAGS
import cv2
import numpy as np

# Configure logging to track forensic steps
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

class ImageForensicsAnalyzer:
    def __init__(self, software_signatures=None):
        self.signatures = software_signatures or [
            'adobe', 'photoshop', 'gimp', 'canva', 'firefly',
            'stability', 'midjourney', 'dall-e', 'stable diffusion'
        ]

    def extract_metadata_signatures(self, file_path: str) -> bool:
        """Inspects both file info blocks (PNG) and EXIF data for editing/AI signatures."""
        logger.info(f"Executing Metadata Analysis on: {file_path}")
        try:
            with Image.open(file_path) as img:
                # 1. Check img.info (crucial for PNG text chunks, software tags, embedded prompts)
                for key, value in img.info.items():
                    if any(sig in str(value).lower() for sig in self.signatures):
                        logger.warning(f"Metadata signature hit found: {key} -> {value}")
                        return True

                # 2. Check EXIF data (standard for JPEGs and traditional image formats)
                exif_data = img._getexif()
                if exif_data:
                    for tag_id, value in exif_data.items():
                        tag_name = TAGS.get(tag_id, tag_id)
                        if tag_name == 'Software' and any(
                            sig in str(value).lower() for sig in self.signatures
                        ):
                            logger.warning(f"EXIF tag signature hit found: Software -> {value}")
                            return True
                            
            logger.info("Metadata verification passed clean.")
            return False
            
        except Exception as e:
            logger.error(f"Could not parse metadata for {file_path}: {e}")
            return False

    def analyze_comprehensive(self, file_path: str) -> dict:
        """Runs a complete check combining metadata footprints and pixel anomalies."""
        report = {
            "file_path": file_path,
            "is_suspicious": False,
            "flags": []
        }

        # Step 1: Metadata Signature Check
        has_bad_metadata = self.extract_metadata_signatures(file_path)
        if has_bad_metadata:
            report["is_suspicious"] = True
            report["flags"].append("Editing/AI software signature detected in metadata.")

        # Step 2: Pixel Noise & Edge Variance Check (OpenCV fallback for stripped metadata)
        cv_img = cv2.imread(file_path)
        if cv_img is not None:
            gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
            laplacian = cv2.Laplacian(gray, cv2.CV_64F)
            
            # Local block variation check for pasted sections
            h, w = gray.shape
            block_size = 64
            block_variances = []
            
            for y in range(0, h - block_size, block_size):
                for x in range(0, w - block_size, block_size):
                    block = laplacian[y:y+block_size, x:x+block_size]
                    block_variances.append(np.var(block))
                    
            if block_variances:
                var_std = np.std(block_variances)
                if var_std > (np.mean(block_variances) * 1.5):
                    report["is_suspicious"] = True
                    report["flags"].append("Localized texture/noise inconsistency detected (possible paste/splice).")

        return report

# ==========================================
# 🧪 EXECUTION
# ==========================================
if __name__ == "__main__":
    analyzer = ImageForensicsAnalyzer()
    test_path = r"D:\Jamaludeen\Project-1-AI-Image-detection\images\manual_Edited_img.jpg"
    
    result = analyzer.analyze_comprehensive(test_path)
    print("\n--- Final Forensic Report ---")
    print(f"🚨 Is Suspicious: {result['is_suspicious']}")
    for flag in result['flags']:
        print(f"   - {flag}")

2026-10-27 17:33:56,288 - INFO - Executing Metadata Analysis on: D:\Jamaludeen\Project-1-AI-Image-detection\images\manual_Edited_img.jpg
2026-10-27 17:33:56,293 - WARNING - Metadata signature hit found: xmp -> b"<?xpacket begin='\xef\xbb\xbf' id='W5M0MpCehiHzreSzNTczkc9d'?>\n<x:xmpmeta xmlns:x='adobe:ns:meta/'>\n<rdf:RDF xmlns:rdf='http://www.w3.org/1999/02/22-rdf-syntax-ns#'>\n\n <rdf:Description rdf:about=''\n  xmlns:Attrib='http://ns.attribution.com/ads/1.0/'>\n  <Attrib:Ads>\n   <rdf:Seq>\n    <rdf:li rdf:parseType='Resource'>\n     <Attrib:Created>2026-10-26</Attrib:Created>\n     <Attrib:Data>{&quot;doc&quot;:&quot;DAHZFRKKvTg&quot;,&quot;user&quot;:&quot;UAF5i1GD2uw&quot;,&quot;brand&quot;:&quot;BAF5izXaPwY&quot;}</Attrib:Data>\n     <Attrib:ExtId>4a58438b-b5ab-4665-97c1-caa12f13a92e</Attrib:ExtId>\n     <Attrib:FbId>525265914179580</Attrib:FbId>\n     <Attrib:TouchType>2</Attrib:TouchType>\n    </rdf:li>\n   </rdf:Seq>\n  </Attrib:Ads>\n </rdf:Description>\n\n <rdf:Description 


--- Final Forensic Report ---
🚨 Is Suspicious: True
   - Editing/AI software signature detected in metadata.


In [14]:
import logging
from PIL import Image
from PIL.ExifTags import TAGS
import cv2
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger(__name__)

class EnterpriseMetadataScanner:
    def __init__(self, custom_signatures=None):
        # Comprehensive enterprise signature registry
        self.signatures = custom_signatures or [
            # Desktop Editors
            'adobe', 'photoshop', 'lightroom', 'gimp', 'affinity', 'pixelmator', 
            'corel', 'paint.net', 'capture one', 'luminar', 'photopea',
            # Mobile/Web Editors
            'canva', 'picsart', 'snapseed', 'facetune', 'vsco', 'pixlr', 'meitu',
            # AI Generators & Frameworks
            'firefly', 'stability', 'midjourney', 'dall-e', 'openai', 'stable diffusion',
            'automatic1111', 'comfyui', 'fooocus', 'invokeai', 'flux', 'leonardo', 'ideogram'
        ]

    def deep_scan_metadata(self, file_path: str) -> list:
        flags = []
        try:
            with Image.open(file_path) as img:
                
                # 1. Scan PNG Info Chunks & AI Generation Parameters (Prompt/Workflow text chunks)
                if hasattr(img, 'info') and img.info:
                    for key, value in img.info.items():
                        val_str = str(value).lower()
                        # Check for software signatures or AI generation parameters
                        if any(sig in val_str for sig in self.signatures) or key.lower() in ['parameters', 'workflow', 'prompt', 'generator']:
                            flags.append(f"Metadata hit in PNG info chunk [{key}]: {str(value)[:100]}...")

                # 2. Scan Standard EXIF Data
                exif_data = img._getexif()
                if exif_data:
                    for tag_id, value in exif_data.items():
                        tag_name = TAGS.get(tag_id, tag_id)
                        val_str = str(value).lower()
                        if tag_name in ['Software', 'HostComputer', 'ImageDescription', 'Artist'] and any(sig in val_str for sig in self.signatures):
                            flags.append(f"EXIF tag hit [{tag_name}]: {value}")

                # 3. Scan XMP Metadata (XML packets embedded in image headers)
                xmp_data = img.getxmp() if hasattr(img, 'getxmp') else None
                if xmp_data:
                    xmp_str = str(xmp_data).lower()
                    for sig in self.signatures:
                        if sig in xmp_str:
                            flags.append(f"XMP Packet signature match found: '{sig}' detected in XML metadata structure.")
                            break

        except Exception as e:
            logger.error(f"Error parsing metadata layers for {file_path}: {e}")

        return flags

# ==========================================
# 🧪 TEST SCRIPT
# ==========================================
if __name__ == "__main__":
    scanner = EnterpriseMetadataScanner()
    for i in range(2,9):
        test_image = f"D:\Jamaludeen\Project-1-AI-Image-detection\images\image{i}f.png"
        if i in [7,8]:
            test_image = f"D:\Jamaludeen\Project-1-AI-Image-detection\images\image{i}f.jpeg"
        if i==2:
                    test_image = f"D:\Jamaludeen\Project-1-AI-Image-detection\images\image{i}f.jpg"
    
        detected_flags = scanner.deep_scan_metadata(test_image)
        print("\n--- Deep Metadata Scan Results ---")
        if detected_flags:
            for f in detected_flags:
                print(f"🚨 {f}")
        else:
            print("✅ Clean: No known software or AI signatures found in metadata layers.")

<>:66: SyntaxWarning: invalid escape sequence '\J'
<>:68: SyntaxWarning: invalid escape sequence '\J'
<>:70: SyntaxWarning: invalid escape sequence '\J'
<>:66: SyntaxWarning: invalid escape sequence '\J'
<>:68: SyntaxWarning: invalid escape sequence '\J'
<>:70: SyntaxWarning: invalid escape sequence '\J'
C:\Users\HP-600 G3\AppData\Local\Temp\ipykernel_8740\3267108615.py:66: SyntaxWarning: invalid escape sequence '\J'
  test_image = f"D:\Jamaludeen\Project-1-AI-Image-detection\images\image{i}f.png"
C:\Users\HP-600 G3\AppData\Local\Temp\ipykernel_8740\3267108615.py:68: SyntaxWarning: invalid escape sequence '\J'
  test_image = f"D:\Jamaludeen\Project-1-AI-Image-detection\images\image{i}f.jpeg"
C:\Users\HP-600 G3\AppData\Local\Temp\ipykernel_8740\3267108615.py:70: SyntaxWarning: invalid escape sequence '\J'
  test_image = f"D:\Jamaludeen\Project-1-AI-Image-detection\images\image{i}f.jpg"



--- Deep Metadata Scan Results ---
✅ Clean: No known software or AI signatures found in metadata layers.

--- Deep Metadata Scan Results ---
✅ Clean: No known software or AI signatures found in metadata layers.

--- Deep Metadata Scan Results ---
✅ Clean: No known software or AI signatures found in metadata layers.

--- Deep Metadata Scan Results ---
✅ Clean: No known software or AI signatures found in metadata layers.

--- Deep Metadata Scan Results ---
✅ Clean: No known software or AI signatures found in metadata layers.

--- Deep Metadata Scan Results ---
✅ Clean: No known software or AI signatures found in metadata layers.

--- Deep Metadata Scan Results ---
✅ Clean: No known software or AI signatures found in metadata layers.
